### Binary classification using Deep Neural Networks Example: Classify movie reviews into positive" reviews and "negative" reviews, just based on the text content of the reviews. Use IMDB dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
print("All libraries imported successfully.")

In [ ]:
# Hyperparameters
NUM_WORDS = 10000 # vocabulary size (top 10k frequent words)
MAX_LEN = 250 # pad/truncate each review to 250 tokens
# Load IMDB dataset (50k reviews: 25k train, 25k test)
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=NUM_WORDS)
print(f"Training samples : {len(x_train)}")
print(f"Test samples : {len(x_test)}")
print(f"Sample label : {y_train[0]} (1=positive, 0=negative)")
print(f"Sample review : {x_train[0][:10]} ...")

In [ ]:
# Pad shorter reviews with zeros; truncate longer ones to MAX_LEN
x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding='post', truncating='post')
print(f"x_train shape : {x_train.shape}") # (25000, 250)
print(f"x_test shape : {x_test.shape}") # (25000, 250)

In [ ]:
# ── Model Architecture ──────────────────────────────────────────────
# Embedding → BiLSTM(64) → BiLSTM(32) → Dense(1, sigmoid)
# ────────────────────────────────────────────────────────────────────
model = Sequential([
# 1. Embedding layer — maps word indices to dense vectors
Embedding(input_dim=NUM_WORDS, output_dim=128, input_shape=(MAX_LEN,)),
# 2. First Bidirectional LSTM — returns full sequence for next layer
Bidirectional(LSTM(64, return_sequences=True)),
Dropout(0.3),
# 3. Second Bidirectional LSTM — returns final hidden state only
Bidirectional(LSTM(32)),
Dropout(0.3),
# 4. Output layer — sigmoid → probability of positive sentiment
Dense(1, activation='sigmoid')
])
model.summary()

In [ ]:
# Binary cross-entropy is the standard loss for binary classification
model.compile(
loss='binary_crossentropy',
optimizer='adam',
metrics=['accuracy']
)
print("Model compiled successfully.")

In [ ]:
# Early stopping stops training if val_loss stops improving (saves time)
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
history = model.fit(
x_train, y_train,
epochs=10,
batch_size=128,
validation_split=0.2, # 20% of training data used for validation
callbacks=[early_stop]
)

In [ ]:
loss, acc = model.evaluate(x_test, y_test, batch_size=128)
print(f"Test accuracy : {acc:.4f}")
print(f"Test loss : {loss:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Accuracy plot
axes[0].plot(history.history['accuracy'],

linestyle='-', marker='o', label='Train Accuracy')

axes[0].plot(history.history['val_accuracy'],

linestyle='--', marker='s', label='Val Accuracy')

axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
# Loss plot
axes[1].plot(history.history['loss'],

linestyle='-', marker='o', label='Train Loss')
axes[1].plot(history.history['val_loss'],

linestyle='--', marker='s', label='Val Loss')

axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
plt.tight_layout()
plt.show()